In [1]:
import importlib
import tiktoken
print(f"titoken version : {importlib.metadata.version("tiktoken")}")

titoken version : 0.8.0


In [2]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
    "of someunknownPlace."
)
tokenizer = tiktoken.get_encoding("gpt2")
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)
strings = tokenizer.decode(integers)
print(strings)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]
Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.


In [5]:
import os
import requests
with open("datas/the-verdict.txt","r",encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
enc_sample = enc_text[:50]
context_size = 4
x = enc_sample[0:context_size]
y = enc_sample[1:context_size+1]
print(f"x: {x}")
print(f"y:     {y}")

x: [40, 367, 2885, 1464]
y:     [367, 2885, 1464, 1807]


In [10]:
import torch
from torch.utils.data import Dataset, DataLoader
# self-supervised data set을 확보
class GPTDataserV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        assert len(token_ids) > max_length, "토큰의 길이는 max_length보다 반드시 커야 합니다"
        for i in range(0, len(token_ids)-max_length,stride):
            input_chunk = token_ids[i:i+max_length]
            target_chunk = token_ids[i+1:i+1+max_length]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [11]:
def create_dataloader_v1(txt, tokenizer, max_length=256, stride=128, batch_size=4, shuffle=True,drop_last=True,num_workers=0):
    # dataset을 만든다
    dataset = GPTDataserV1(text, tokenizer, max_length, stride) 
    # dataloader를 만들어 리턴
    dataloader = DataLoader(
        dataset, 
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )
    return dataloader

In [14]:
tokenizer = tiktoken.get_encoding("gpt2")
dataloader = create_dataloader_v1(
    txt=raw_text,
    tokenizer=tokenizer,
    max_length=4,
    stride=1,
    batch_size=1,
    shuffle=False,
    drop_last=False,
    num_workers=0
)

batch = iter(dataloader)
first_batch = next(batch)
print(first_batch)
second_batch = next(batch)
print(second_batch)


[tensor([[15496,    11,   466,   345]]), tensor([[ 11, 466, 345, 588]])]
[tensor([[ 11, 466, 345, 588]]), tensor([[ 466,  345,  588, 8887]])]


In [16]:
input_ids = torch.tensor([2,3,5,1])

vocab_size = 6
output_dim = 3

embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
print(embedding_layer.weight)
token_embeddings = embedding_layer(input_ids)
print(token_embeddings)

Parameter containing:
tensor([[ 0.1709,  0.4397, -0.5030],
        [-1.4195, -0.1118,  0.9399],
        [ 1.3488, -0.8428,  0.0320],
        [-0.5134,  0.1986, -0.6536],
        [-1.0703,  0.2078,  0.7278],
        [-0.8918, -0.1229,  0.4179]], requires_grad=True)
tensor([[ 1.3488, -0.8428,  0.0320],
        [-0.5134,  0.1986, -0.6536],
        [-0.8918, -0.1229,  0.4179],
        [-1.4195, -0.1118,  0.9399]], grad_fn=<EmbeddingBackward0>)


In [25]:
tokenizer = tiktoken.get_encoding("gpt2")
dataset = GPTDataserV1(txt=raw_text, tokenizer=tokenizer, max_length=4, stride=4)
dataloader = DataLoader(
    dataset=dataset,
    batch_size=8,
    shuffle=False,
    drop_last=True,
    num_workers=0
)

data_iter = iter(dataloader)
input_ids, target_ids = next(data_iter)
print(input_ids)
print(input_ids.shape)
print(target_ids)
print(target_ids.shape)

# embedding을 하는데 token embeding이랑 pos embedding을 만들어서 더한다
vocab_size = 50257
output_dim = 256
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

data_iter = iter(dataloader)
input_ids, target_ids = next(data_iter)
print(input_ids.shape)
token_embeddings = token_embedding_layer(input_ids)
print(token_embeddings.shape)

pos_embedding_layer = torch.nn.Embedding(context_size, output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(4))
print(pos_embeddings.shape)

input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)




tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
torch.Size([8, 4])
tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])
torch.Size([8, 4])
torch.Size([8, 4])
torch.Size([8, 4, 256])
torch.Size([4, 256])
torch.Size([8, 4, 256])
